# 🔬 Georgia Clinical Trial Navigator (GCTN)
**Instructions:** Click **Run All** to launch the app. To reopen after closing the window, re-run only Cell 9.

In [ ]:
# ── Cell 1: Imports & API Key ────────────────────────────────────
import tkinter as tk
from tkinter import messagebox
import urllib.request
import urllib.parse
import json
import os

def load_api_key():
    try:
        base_dir = os.path.dirname(os.path.abspath(__file__))
    except NameError:
        base_dir = os.getcwd()
    env_path = os.path.join(base_dir, '.env')
    if os.path.exists(env_path):
        with open(env_path, 'r') as f:
            for line in f:
                line = line.strip()
                if line.startswith('CLAUDE_API_KEY='):
                    return line.split('=', 1)[1].strip()
    return ''

CLAUDE_API_KEY = load_api_key()
print(f'API Key loaded: {"Yes ✅" if CLAUDE_API_KEY else "No ⚠ — .env file not found in: " + os.getcwd()}')


In [ ]:
# ── Cell 2: Color Theme & Fonts ──────────────────────────────────
CLR_BG         = '#1e2a3a'
CLR_PANEL      = '#253447'
CLR_ENTRY_BG   = '#f0f4f8'
CLR_ENTRY_FG   = '#1a1a2e'
CLR_LABEL_FG   = '#d0dce8'
CLR_HINT_FG    = '#6a8fa8'
CLR_RESULT_BG  = '#f8fafc'
CLR_RESULT_FG  = '#1a2533'
CLR_BTN_SEARCH = '#1a7abf'
CLR_BTN_CLEAR  = '#b03a2e'
CLR_BTN_AI     = '#1a8c5a'
CLR_BTN_CLOSE  = '#4a5568'
CLR_BTN_FG     = '#ffffff'
CLR_STATUS_BG  = '#141e2b'
CLR_STATUS_FG  = '#7fa8c4'
CLR_OK         = '#2ecc71'
CLR_WARN       = '#e67e22'
CLR_ERR        = '#e74c3c'

FONT_LABEL      = ('Helvetica', 11)
FONT_LABEL_BOLD = ('Helvetica', 11, 'bold')
FONT_HINT       = ('Helvetica', 9)
FONT_BTN        = ('Helvetica', 11, 'bold')
FONT_ENTRY      = ('Helvetica', 11)
FONT_RESULT     = ('Helvetica', 10)
FONT_STATUS     = ('Helvetica', 9)
FONT_TITLE      = ('Helvetica', 13, 'bold')

print('Theme loaded. ✅')


In [ ]:
# ── Cell 3: Plain-Language Dictionary ───────────────────────────
plain_language_terms = {
    'nsclc': 'Non-small cell lung cancer, the most common type of lung cancer.',
    'sclc': 'Small cell lung cancer, a faster-growing type of lung cancer.',
    'egfr': 'A gene change in lung cancer that may respond to targeted therapy.',
    'alk': 'A gene change in lung cancer that can be treated with targeted drugs.',
    'kras': 'A common gene change found in lung cancer.',
    'pd-1': 'A protein on immune cells. Some cancers block PD-1 to avoid attack.',
    'pd-l1': 'A protein on cancer cells that helps them hide from the immune system.',
    'her2': 'HER2-positive breast cancer, which may respond to targeted therapy.',
    'er': 'Estrogen receptor-positive breast cancer.',
    'pr': 'Progesterone receptor-positive breast cancer.',
    'tnbc': 'Triple-negative breast cancer, which does not have hormone receptors.',
    'hormone receptor': 'A feature that affects how breast cancer grows and is treated.',
    'msi': 'MSI-high colon cancer, which may respond to immunotherapy.',
    'ras': 'A gene change that affects colon cancer treatment options.',
    'braf': 'A gene change that affects colon cancer growth and treatment.',
    'metastatic': 'The cancer has spread to other parts of the body.',
    'resectable': 'The tumor can be removed with surgery.',
    'unresectable': 'The tumor cannot be removed with surgery.',
    'locally advanced': 'The cancer has grown nearby but has not spread far away.',
    'brca': 'A gene change linked to some pancreatic and breast cancers.',
    'gleason': 'A score that describes how aggressive prostate cancer is.',
    'psa': 'A blood test used to help detect prostate cancer.',
    'androgen': 'Male hormones that can help prostate cancer grow.',
    'castration-resistant': 'Cancer that grows even when testosterone is low.',
    'investigational': 'A new drug or treatment that is still being studied.',
    'efficacy': 'How well a treatment works.',
    'tolerability': 'How well a person can handle the side effects.',
    'monotherapy': 'Using one drug alone.',
    'combination therapy': 'Using more than one drug together.',
    'randomized': 'Participants are assigned to groups by chance.',
    'double-blind': 'Neither the patient nor the doctor knows which treatment is given.',
    'open-label': 'Everyone knows which treatment is being given.',
    'placebo': 'A treatment with no active drug.',
    'inclusion criteria': 'Requirements a person must meet to join a study.',
    'exclusion criteria': 'Reasons a person cannot join a study.',
    'phase 1': 'A study phase that tests safety and dose.',
    'phase 2': 'A study phase that tests whether the treatment works.',
    'phase 3': 'A study phase that compares treatments.',
    'phase 4': 'A study phase that looks at long-term effects.',
    'primary endpoint': 'The main result the study is measuring.',
    'secondary endpoint': 'Extra results the study is also tracking.',
    'cohort': 'A group of patients in a study.',
    'arm': 'A treatment group in a clinical trial.',
    'adverse event': 'A side effect or unwanted reaction during the study.',
    'biomarker': 'A measurable sign in the body used to track disease or treatment response.',
}
print(f'Dictionary loaded: {len(plain_language_terms)} terms. ✅')


In [ ]:
# ── Cell 4: Georgia Regions & Centers ───────────────────────────
regions = {
    'augusta': 'East Georgia',
    'atlanta': 'Metro Atlanta',
    'decatur': 'Metro Atlanta',
    'marietta': 'Metro Atlanta',
    'alpharetta': 'Metro Atlanta',
    'roswell': 'Metro Atlanta',
    'sandy springs': 'Metro Atlanta',
    'savannah': 'South Georgia',
    'albany': 'South Georgia',
    'valdosta': 'South Georgia',
    'brunswick': 'South Georgia',
    'columbus': 'West Georgia',
    'lagrange': 'West Georgia',
    'rome': 'North Georgia',
    'gainesville': 'North Georgia',
    'macon': 'Central Georgia',
    'warner robins': 'Central Georgia',
}

centers = {
    'East Georgia': ['Georgia Cancer Center (Augusta University)'],
    'Metro Atlanta': [
        'Emory Winship Cancer Institute',
        'Northside Hospital Cancer Institute',
        'Piedmont Atlanta',
        'Wellstar Health System',
    ],
    'South Georgia': [
        "St. Joseph's/Candler (Savannah)",
        'Phoebe Putney Memorial Hospital (Albany)',
    ],
    'West Georgia': ['Piedmont Columbus Regional'],
    'North Georgia': ['AdventHealth Rome', 'Northeast Georgia Medical Center (Gainesville)'],
    'Central Georgia': ['Navicent Health / Atrium Health (Macon)'],
}
print('Regions loaded. ✅')


In [ ]:
# ── Cell 5: Helper Functions ─────────────────────────────────────
def find_region(city):
    city = city.lower().strip()
    for key in regions:
        if key in city:
            return regions[key]
    return None

def check_eligibility(age, cancer_type):
    cancer_type = cancer_type.lower()
    if age < 18:
        return 'Most clinical trials require participants to be 18 or older.'
    valid_types = ['lung', 'colon', 'pancreatic', 'prostate', 'breast']
    if cancer_type not in valid_types:
        return 'This tool checks lung, colon, pancreatic, prostate, and breast cancer. Your cancer type may still have trials — results shown are statewide Georgia.'
    return f'You may be eligible for trials related to {cancer_type} cancer.'

def explain_subtype(subtype):
    subtype = subtype.lower().strip()
    if subtype in plain_language_terms:
        return plain_language_terms[subtype]
    if subtype == '':
        return 'No specific subtype entered.'
    return 'This subtype is not in the dictionary, but some trials are subtype-specific. Ask your doctor for more details.'

def explain_terms_in_text(text):
    text_lower = text.lower()
    explanations = []
    for term, definition in plain_language_terms.items():
        if term in text_lower:
            explanations.append(f'{term.upper()}: {definition}')
    if not explanations:
        explanations.append('No specific medical terms recognized.')
    return explanations

def set_status(message, color=None):
    if color is None:
        color = CLR_STATUS_FG
    status_label.config(text=f'  {message}', fg=color)
    root.update_idletasks()

def make_button(parent, text, command, color, **kwargs):
    return tk.Button(
        parent, text=text, command=command,
        bg=color, fg=CLR_BTN_FG, font=FONT_BTN,
        relief=tk.FLAT, activebackground=color,
        activeforeground=CLR_BTN_FG, cursor='hand2',
        padx=16, pady=7, **kwargs,
    )

print('Helper functions loaded. ✅')


In [ ]:
# ── Cell 6: ClinicalTrials.gov API ───────────────────────────────
CTGOV_API = 'https://clinicaltrials.gov/api/v2/studies'

def build_query(cancer_type, subtype, city):
    condition = cancer_type + ' cancer'
    if subtype:
        condition += ' ' + subtype
    params = {
        'query.cond': condition,
        'query.locn': 'Georgia',
        'filter.overallStatus': 'RECRUITING',
        'aggFilters': 'studyType:int',
        'pageSize': 10,
        'format': 'json',
        'fields': 'NCTId,BriefTitle,BriefSummary,Phase,EligibilityCriteria,OverallStatus,LocationCity,LocationFacility,MinimumAge,MaximumAge,Condition',
    }
    return CTGOV_API + '?' + urllib.parse.urlencode(params)

def fetch_trials(cancer_type, subtype, city):
    url = build_query(cancer_type, subtype, city)
    try:
        req = urllib.request.Request(url, headers={'User-Agent': 'GCTN/1.0'})
        with urllib.request.urlopen(req, timeout=15) as response:
            data = json.loads(response.read().decode('utf-8'))
    except Exception as e:
        return None, f'Could not connect to ClinicalTrials.gov: {e}'
    studies = data.get('studies', [])
    if not studies:
        return [], None
    results = []
    for study in studies:
        proto        = study.get('protocolSection', {})
        id_mod       = proto.get('identificationModule', {})
        desc_mod     = proto.get('descriptionModule', {})
        design_mod   = proto.get('designModule', {})
        elig_mod     = proto.get('eligibilityModule', {})
        contacts_mod = proto.get('contactsLocationsModule', {})
        nct_id    = id_mod.get('nctId', 'N/A')
        title     = id_mod.get('briefTitle', 'No title available')
        summary   = desc_mod.get('briefSummary', 'No summary available.')
        phases    = design_mod.get('phases', [])
        phase_str = ', '.join(phases) if phases else 'Not listed'
        min_age   = elig_mod.get('minimumAge', 'Not specified')
        max_age   = elig_mod.get('maximumAge', 'Not specified')
        locations = contacts_mod.get('locations', [])
        ga_locations = []
        for loc in locations:
            if loc.get('state', '').lower() in ('georgia', 'ga'):
                facility = loc.get('facility', '')
                loc_city = loc.get('city', '')
                if facility or loc_city:
                    ga_locations.append(f'{facility} ({loc_city})' if facility else loc_city)
        results.append({
            'nct_id': nct_id, 'title': title, 'summary': summary,
            'phase': phase_str, 'min_age': min_age, 'max_age': max_age,
            'ga_locations': ga_locations,
        })
    return results, None

def summarize_trial(trial, age):
    lines = [
        f"Trial ID : {trial['nct_id']}",
        f"Title    : {trial['title']}",
        f"Phase    : {trial['phase']}",
        f"Ages     : {trial['min_age']} to {trial['max_age']}",
    ]
    if trial['min_age'] != 'Not specified':
        try:
            min_yr = int(''.join(filter(str.isdigit, trial['min_age'])))
            if age < min_yr:
                lines.append(f"  ⚠  You may not meet the minimum age ({trial['min_age']}) for this trial.")
        except ValueError:
            pass
    sentences = trial['summary'].replace('\n', ' ').split('. ')
    short_summary = '. '.join(sentences[:3]).strip()
    if short_summary and not short_summary.endswith('.'):
        short_summary += '.'
    lines.append(f'\nWhat this trial is about:\n  {short_summary}')
    if trial['ga_locations']:
        lines.append('\nGeorgia locations accepting patients:')
        for loc in trial['ga_locations'][:5]:
            lines.append(f'  - {loc}')
    else:
        lines.append('\n  No confirmed Georgia locations listed (check ClinicalTrials.gov for updates).')
    lines.append(f"\n  More info: https://clinicaltrials.gov/study/{trial['nct_id']}")
    lines.append('─' * 70)
    return '\n'.join(lines)

print('ClinicalTrials.gov functions loaded. ✅')


In [ ]:
# ── Cell 7: GUI Logic ────────────────────────────────────────────
def on_check():
    name        = entry_name.get().strip()
    age_text    = entry_age.get().strip()
    cancer_type = entry_cancer.get().strip().lower()
    subtype     = entry_subtype.get().strip()
    city        = entry_city.get().strip()

    if not name or not age_text or not cancer_type or not city:
        messagebox.showinfo('Missing Information', 'Please fill in all required fields.')
        return
    if not age_text.isdigit():
        messagebox.showinfo('Invalid Age', 'Please enter a number for age.')
        return

    age = int(age_text)
    eligibility_message = check_eligibility(age, cancer_type)
    subtype_message     = explain_subtype(subtype)
    region              = find_region(city)

    result_lines = [
        f'Results for {name}  |  Age: {age}  |  Cancer Type: {cancer_type.title()}\n',
        f'Eligibility Check:\n  {eligibility_message}\n',
        f'Subtype Information:\n  {subtype_message}\n',
    ]
    if region:
        result_lines.append(f'Nearest Georgia Region: {region}')
        result_lines.append('Nearby Cancer Centers:')
        for c in centers[region]:
            result_lines.append(f'  - {c}')
        result_lines.append('')
    else:
        result_lines.append(f"City '{city}' not found. Showing statewide Georgia results.\n")

    text_result.config(state=tk.NORMAL)
    text_result.delete('1.0', tk.END)
    text_result.insert(tk.END, '\n'.join(result_lines))
    text_result.insert(tk.END, '\nSearching ClinicalTrials.gov — please wait...\n')
    set_status('Searching ClinicalTrials.gov — please wait...', CLR_BTN_SEARCH)
    root.update()

    trials, error = fetch_trials(cancer_type, subtype, city)
    display_trials(trials, error, age)


def display_trials(trials, error, age):
    if error:
        text_result.insert(tk.END, f'\n⚠  {error}\n')
        text_result.insert(tk.END, 'Search manually at: https://clinicaltrials.gov\n')
        set_status('Search failed — check your internet connection.', CLR_ERR)
        text_result.config(state=tk.DISABLED)
        return
    if not trials:
        text_result.insert(tk.END, '\nNo recruiting trials found in Georgia for this cancer type.\n')
        text_result.insert(tk.END, 'Try adjusting your search or visiting https://clinicaltrials.gov\n')
        set_status('No trials found. Try a different search.', CLR_WARN)
        text_result.config(state=tk.DISABLED)
        return
    text_result.insert(tk.END, f"\n{'=' * 70}\n")
    text_result.insert(tk.END, f'  RECRUITING CLINICAL TRIALS IN GEORGIA  ({len(trials)} found)\n')
    text_result.insert(tk.END, f"{'=' * 70}\n\n")
    for trial in trials:
        text_result.insert(tk.END, summarize_trial(trial, age) + '\n')
    text_result.insert(tk.END, '\nNext Steps:\n')
    text_result.insert(tk.END, '  - Talk to your doctor or research nurse about these trials.\n')
    text_result.insert(tk.END, '  - Visit https://clinicaltrials.gov for full eligibility details.\n')
    text_result.config(state=tk.DISABLED)
    set_status(f'Done — {len(trials)} recruiting trials found in Georgia.', CLR_OK)


def on_clear():
    entry_name.delete(0, tk.END)
    entry_age.delete(0, tk.END)
    entry_cancer.delete(0, tk.END)
    entry_subtype.delete(0, tk.END)
    entry_city.delete(0, tk.END)
    text_medical.delete('1.0', tk.END)
    text_result.config(state=tk.NORMAL)
    text_result.delete('1.0', tk.END)
    text_result.config(state=tk.DISABLED)
    set_status('Ready — enter patient information to search.')
    entry_name.focus()


def call_claude_api(medical_text):
    if not CLAUDE_API_KEY:
        return None, (
            'No API key found.\n\n'
            'To enable AI explanations:\n'
            '1. Get a free key at console.anthropic.com\n'
            '2. Create a .env file in your Downloads folder\n'
            '3. Add this line: CLAUDE_API_KEY=sk-ant-your-key-here\n'
            '4. Restart the kernel and Run All.'
        )
    prompt = (
        'You are a patient advocate helping everyday people understand clinical trial documents. '
        'Please explain the following text in warm, clear, plain language as if speaking to someone '
        'with no medical background. Use short paragraphs and a conversational narrative. '
        'Explain medical terms in context rather than defining them in isolation.\n\n'
        f'Clinical trial text:\n{medical_text}'
    )
    payload = json.dumps({
        'model': 'claude-sonnet-4-20250514',
        'max_tokens': 1000,
        'messages': [{'role': 'user', 'content': prompt}]
    }).encode('utf-8')
    req = urllib.request.Request(
        'https://api.anthropic.com/v1/messages',
        data=payload,
        headers={
            'Content-Type': 'application/json',
            'x-api-key': CLAUDE_API_KEY,
            'anthropic-version': '2023-06-01',
        },
        method='POST',
    )
    try:
        with urllib.request.urlopen(req, timeout=30) as response:
            data = json.loads(response.read().decode('utf-8'))
            return data['content'][0]['text'], None
    except urllib.error.HTTPError as e:
        return None, f"API error {e.code}: {e.read().decode('utf-8')}"
    except Exception as e:
        return None, f'Could not reach Claude API: {e}'


def on_translate():
    medical_text = text_medical.get('1.0', tk.END).strip()
    if not medical_text:
        messagebox.showinfo('Missing Text', 'Please paste clinical trial text to explain.')
        return
    popup = tk.Toplevel(root)
    popup.title('Plain-Language Explanation')
    popup.geometry('680x520')
    popup.resizable(True, True)
    popup.config(bg=CLR_BG)
    tk.Label(popup, text='Plain-Language Explanation  (Powered by Claude AI)',
             font=FONT_TITLE, bg=CLR_BG, fg=CLR_LABEL_FG, pady=14).pack()
    frame_pop = tk.Frame(popup, bg=CLR_BG)
    frame_pop.pack(fill=tk.BOTH, expand=True, padx=14, pady=(0, 4))
    pop_scrollbar = tk.Scrollbar(frame_pop)
    pop_scrollbar.pack(side=tk.RIGHT, fill=tk.Y)
    pop_text = tk.Text(frame_pop, wrap=tk.WORD, yscrollcommand=pop_scrollbar.set,
                       font=('Helvetica', 11), padx=14, pady=12, relief=tk.FLAT,
                       bg=CLR_RESULT_BG, fg=CLR_RESULT_FG)
    pop_text.pack(fill=tk.BOTH, expand=True)
    pop_scrollbar.config(command=pop_text.yview)
    pop_text.insert(tk.END, 'Asking Claude to explain this — please wait...\n')
    popup.update()
    set_status('Waiting for Claude AI response...', CLR_BTN_SEARCH)
    narrative, error = call_claude_api(medical_text)
    pop_text.config(state=tk.NORMAL)
    pop_text.delete('1.0', tk.END)
    if error:
        pop_text.insert(tk.END, f'⚠  {error}\n\n')
        pop_text.insert(tk.END, '─' * 50 + '\n')
        pop_text.insert(tk.END, 'Dictionary definitions for recognized terms:\n\n')
        for e in explain_terms_in_text(medical_text):
            pop_text.insert(tk.END, f'  • {e}\n\n')
        set_status('AI unavailable — showed dictionary fallback.', CLR_WARN)
    else:
        pop_text.insert(tk.END, narrative)
        set_status('AI explanation complete.', CLR_OK)
    pop_text.config(state=tk.DISABLED)
    make_button(popup, 'Close', popup.destroy, CLR_BTN_CLOSE).pack(pady=12)


print('GUI logic loaded. ✅')


In [ ]:
# ── Cell 8: GUI Layout Helpers ───────────────────────────────────
def styled_label_frame(parent, title):
    return tk.LabelFrame(parent, text=f'  {title}  ', font=FONT_LABEL_BOLD,
                         bg=CLR_PANEL, fg=CLR_LABEL_FG, padx=14, pady=12,
                         relief=tk.GROOVE, bd=2)

def styled_label(parent, text, hint=False):
    return tk.Label(parent, text=text,
                    font=FONT_HINT if hint else FONT_LABEL,
                    bg=CLR_PANEL,
                    fg=CLR_HINT_FG if hint else CLR_LABEL_FG)

def styled_entry(parent, width):
    return tk.Entry(parent, width=width, font=FONT_ENTRY,
                    bg=CLR_ENTRY_BG, fg=CLR_ENTRY_FG,
                    insertbackground=CLR_ENTRY_FG, relief=tk.FLAT, bd=4)

print('Layout helpers loaded. ✅')


In [ ]:
# ── Cell 9: Launch the App ───────────────────────────────────────
# To reopen the app after closing the window, re-run only this cell.

root = tk.Tk()
root.title('Georgia Clinical Trial Navigator (GCTN)')
root.resizable(True, True)
root.minsize(760, 650)
root.config(bg=CLR_BG)
root.grid_columnconfigure(0, weight=1)

# Header
header = tk.Frame(root, bg='#132033', pady=14)
header.grid(row=0, column=0, sticky='ew')
tk.Label(header, text='🔬  Georgia Clinical Trial Navigator',
         font=('Helvetica', 16, 'bold'), bg='#132033', fg='#ffffff').pack(side=tk.LEFT, padx=20)
tk.Label(header, text='Helping Georgia patients find cancer clinical trials',
         font=('Helvetica', 10), bg='#132033', fg='#7fa8c4').pack(side=tk.LEFT, padx=6)

# Patient Information
frame_input = styled_label_frame(root, 'Patient Information')
frame_input.grid(row=1, column=0, padx=14, pady=(10, 6), sticky='ew')
styled_label(frame_input, 'Full Name:').grid(row=0, column=0, sticky='e', pady=5)
entry_name = styled_entry(frame_input, 30)
entry_name.grid(row=0, column=1, sticky='w', pady=5, padx=8)
styled_label(frame_input, 'Age:').grid(row=1, column=0, sticky='e', pady=5)
entry_age = styled_entry(frame_input, 10)
entry_age.grid(row=1, column=1, sticky='w', pady=5, padx=8)
styled_label(frame_input, 'Cancer Type:').grid(row=2, column=0, sticky='e', pady=5)
entry_cancer = styled_entry(frame_input, 20)
entry_cancer.grid(row=2, column=1, sticky='w', pady=5, padx=8)
styled_label(frame_input, 'lung · colon · pancreatic · prostate · breast', hint=True).grid(row=2, column=2, sticky='w')
styled_label(frame_input, 'Subtype (optional):').grid(row=3, column=0, sticky='e', pady=5)
entry_subtype = styled_entry(frame_input, 20)
entry_subtype.grid(row=3, column=1, sticky='w', pady=5, padx=8)
styled_label(frame_input, 'e.g. NSCLC, HER2, TNBC, MSI', hint=True).grid(row=3, column=2, sticky='w')
styled_label(frame_input, 'Georgia City:').grid(row=4, column=0, sticky='e', pady=5)
entry_city = styled_entry(frame_input, 20)
entry_city.grid(row=4, column=1, sticky='w', pady=5, padx=8)
frame_buttons = tk.Frame(frame_input, bg=CLR_PANEL)
frame_buttons.grid(row=5, column=0, columnspan=3, pady=14)
make_button(frame_buttons, '🔍  Search Clinical Trials', on_check, CLR_BTN_SEARCH).pack(side=tk.LEFT, padx=10)
make_button(frame_buttons, '🔄  Clear / New Search', on_clear, CLR_BTN_CLEAR).pack(side=tk.LEFT, padx=10)

# Translate
frame_translate = styled_label_frame(root, 'Translate Medical Terms  (AI-Powered)')
frame_translate.grid(row=2, column=0, padx=14, pady=6, sticky='ew')
key_status = '✅  Claude AI ready' if CLAUDE_API_KEY else '⚠  Claude AI not configured — add .env file with CLAUDE_API_KEY to enable'
key_color  = CLR_OK if CLAUDE_API_KEY else CLR_ERR
tk.Label(frame_translate, text=key_status, fg=key_color,
         font=('Helvetica', 9, 'italic'), bg=CLR_PANEL).grid(row=0, column=0, columnspan=2, sticky='w', pady=(0, 8))
styled_label(frame_translate, 'Paste clinical trial text:').grid(row=1, column=0, sticky='nw', pady=4)
text_medical = tk.Text(frame_translate, width=64, height=5, font=FONT_ENTRY,
                       bg=CLR_ENTRY_BG, fg=CLR_ENTRY_FG,
                       insertbackground=CLR_ENTRY_FG, relief=tk.FLAT, bd=4)
text_medical.grid(row=1, column=1, padx=8, pady=4)
make_button(frame_translate, '💬  Explain in Plain Language', on_translate, CLR_BTN_AI).grid(
    row=2, column=0, columnspan=2, pady=10)

# Results
frame_results = styled_label_frame(root, 'Results')
frame_results.grid(row=3, column=0, padx=14, pady=6, sticky='nsew')
root.grid_rowconfigure(3, weight=1)
scrollbar = tk.Scrollbar(frame_results)
scrollbar.pack(side=tk.RIGHT, fill=tk.Y)
text_result = tk.Text(frame_results, width=80, height=18,
                      yscrollcommand=scrollbar.set, wrap=tk.WORD,
                      font=FONT_RESULT, padx=10, pady=8, relief=tk.FLAT,
                      bg=CLR_RESULT_BG, fg=CLR_RESULT_FG, state=tk.DISABLED)
text_result.pack(fill=tk.BOTH, expand=True)
scrollbar.config(command=text_result.yview)

# Status bar
status_label = tk.Label(root, text='  Ready — enter patient information to search.',
                        anchor='w', fg=CLR_STATUS_FG, bg=CLR_STATUS_BG,
                        font=FONT_STATUS, pady=5)
status_label.grid(row=4, column=0, sticky='ew')

print('App launched! ✅')
root.mainloop()
